# Mythos 6 — Colab training notebook

Thin wrapper around the `mythos6` repo's `scripts/`/`src/mythos` pipeline. See `ARCHITECTURE.md` and `MILESTONES.md` in the repo before changing anything here.

Works on free-tier T4 or Pro/Pro+ A100 — `train.py` auto-selects bf16 on A100 and fp16+GradScaler on T4 (see ARCHITECTURE.md sec 6.1), no notebook changes needed either way. Colab's runtime disk (and, unless Drive is mounted, its local files) is wiped on disconnect — this notebook mounts Drive and writes checkpoints/shards there specifically so a disconnect doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/mythos6"
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/Bentolamb/mythos6.git"
WORK_DIR = "/content/mythos6"

if not os.path.exists(WORK_DIR):
    !git clone $REPO_URL $WORK_DIR
%cd $WORK_DIR
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — check Runtime > Change runtime type."
print(torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

## 1. Tokenizer (train once, persist to Drive)

In [ ]:
TOKENIZER_PATH = f"{DRIVE_DIR}/artifacts/tokenizer.json"
if not os.path.exists(TOKENIZER_PATH):
    !python scripts/train_tokenizer.py --n-docs 200000 --out $TOKENIZER_PATH

## 2. Contamination + dedup gate
Per ARCHITECTURE.md sec 5.3 — a gate, not a formality.

In [ ]:
!python scripts/contamination_scan.py --n-corpus-docs 50000
!python scripts/dedup_scan.py --n-docs 50000

## 3. Pre-tokenize into packed shards (persist to Drive — this step is CPU-only, no GPU quota used)

In [ ]:
SHARD_DIR = f"{DRIVE_DIR}/data/packed"
PRESET = "mythos6-130m-dense"  # start here per MILESTONES.md M2, not the 320m config
SEQ_LEN = 4096
N_SEQUENCES = 400000  # ~1.6B tokens; adjust against the actual token budget in ARCHITECTURE.md sec 6.3

if not os.path.exists(f"{SHARD_DIR}/manifest.json"):
    !python scripts/pretokenize.py --tokenizer $TOKENIZER_PATH --out-dir $SHARD_DIR \
        --seq-len $SEQ_LEN --n-sequences $N_SEQUENCES

## 4. Train
Checkpoints go to Drive, not local Colab disk, so `--resume` works across a disconnect/reconnect (or even a runtime-type change from T4 to A100 mid-project — same shards, same checkpoint format).

In [ ]:
RUN_DIR = f"{DRIVE_DIR}/runs/{PRESET}"

!python -m mythos.train \
    --preset $PRESET \
    --data-dir $SHARD_DIR \
    --out-dir $RUN_DIR \
    --micro-batch-size 8 \
    --grad-accum-steps 16 \
    --max-steps 50000 \
    --save-every 500 \
    --resume